# U05 · Supervisados II — SVM, árboles, ensembles y **cómo elegir bien**

> **Este es el cuaderno más importante del curso.** No va de coleccionar modelos nuevos, sino de la habilidad que de verdad separa a un profesional con criterio de quien sigue la moda: **elegir el mejor modelo sin autoengañarse.**

> ⚠️ Todos los datos son **sintéticos** (generados por código, semilla fija). **No representan pacientes reales.** La primera celda los crea sola: no hay que descargar nada.

**El recorrido, en arco creciente:**
1. **Poner a competir** a varias familias sobre `evento_cv` con una **partición honesta**: logística, SVM, árbol, Random Forest y boosting. Tabla de **AUC en test**.
2. **Validación cruzada**: en lugar de una sola cifra caprichosa, varias rotaciones que dan una estimación **más honesta** y nos dicen **cuánto fiarnos**.
3. **Búsqueda de hiperparámetros** con una **rejilla pequeña**: afinar el modelo sin caer en la trampa de *sobreajustar la propia validación*.
4. **Elegir el mejor** por la **métrica clínica** y evaluarlo **una sola vez** en el test intacto.
5. **Interpretabilidad**: `permutation_importance` (robusta) y una demo **opcional** de **SHAP**.
6. Un cierre breve en **regresión** (`riesgo_cv_10a`), donde el ganador cambia.

**La lección de oro que vamos a comprobar con datos:** *no existe un modelo universalmente mejor.* En esta cohorte, la **humilde logística iguala o gana** en la clasificación de `evento_cv`, mientras que el **Random Forest arrasa** en la regresión de `riesgo_cv_10a`. El mejor modelo **no se adivina: se mide.**

[Abrir en Colab](PENDIENTE_DRIVE)


## 0. Preparamos los datos (esta celda se ejecuta sola)

La primera celda de cada cuaderno **genera los ficheros sintéticos** en la carpeta de trabajo. Ejecútala una vez (▶ o *Mayús+Enter*) y sigue hacia abajo. Recuerda: son datos **inventados**, para aprender, no para decidir sobre personas.


In [ ]:
# === Generación de los datos sintéticos del curso (se ejecuta sola) ===
# TODOS LOS DATOS SON SINTÉTICOS. No representan pacientes reales. Semilla fija -> reproducible.
import os, numpy as np, pandas as pd

def generar_datos_clinicos(carpeta="."):
    """Crea los CSV del curso si no están ya en la carpeta. Devuelve la ruta."""
    if os.path.exists(os.path.join(carpeta, "pacientes.csv")):
        return carpeta
    RNG = np.random.default_rng(2026)   # misma semilla en todo el curso

    # --- Centros ---
    AREAS = ["Valencia","Alicante","Castellón","Madrid","Barcelona","Sevilla","Bilbao","Zaragoza"]
    nc = 24
    tipo = RNG.choice(["hospital","centro de salud"], nc, p=[0.35,0.65])
    camas = np.where(tipo=="hospital", RNG.integers(120,900,nc), RNG.integers(0,25,nc))
    nserv = np.where(tipo=="hospital", RNG.integers(12,40,nc), RNG.integers(3,10,nc))
    centros = pd.DataFrame({"centro_id":[f"C{i:03d}" for i in range(1,nc+1)],"tipo":tipo,
        "area":RNG.choice(AREAS,nc),"camas":camas,"n_servicios":nserv,
        "urgencias_dia_media":np.where(tipo=="hospital",RNG.integers(80,320,nc),RNG.integers(5,40,nc)),
        "ratio_mayores65":RNG.uniform(0.12,0.34,nc).round(3)})
    centros.to_csv(os.path.join(carpeta,"centros.csv"), index=False)

    # --- Pacientes (tabla central) ---
    N = 20000
    sexo = RNG.choice(["M","F"], N, p=[0.49,0.51])
    edad = np.clip(np.where(RNG.random(N)<0.6, RNG.normal(58,15,N), RNG.normal(40,12,N)),18,95).round().astype(int)
    p_act = np.clip(0.28-0.0016*(edad-18),0.06,0.28); p_ex = np.clip(0.10+0.004*(edad-18),0.10,0.40); p_nu = 1-p_act-p_ex
    tabaquismo = np.array([RNG.choice(["nunca","ex","activo"],p=[a,b,c]) for a,b,c in zip(p_nu,p_ex,p_act)])
    actividad = RNG.choice(["baja","media","alta"], N, p=[0.4,0.4,0.2])
    antec = RNG.binomial(1,0.22,N)
    actn = pd.Series(actividad).map({"baja":1.5,"media":0.0,"alta":-1.3}).values
    imc = np.clip(RNG.normal(26.5,4.2,N)+0.03*(edad-50)+actn+RNG.normal(0,1.0,N),15,48).round(1)
    fa = (tabaquismo=="activo").astype(float); fe = (tabaquismo=="ex").astype(float)
    ta_s = np.clip(RNG.normal(120,12,N)+0.45*(edad-45)+0.7*(imc-25)+6*fa+RNG.normal(0,6,N),85,220).round().astype(int)
    ta_d = np.clip(0.55*ta_s+RNG.normal(20,6,N),50,130).round().astype(int)
    glu = np.clip(RNG.normal(100,12,N)+1.0*(imc-25)+0.3*(edad-45)+9*antec+RNG.normal(0,7,N),60,320).round(1)
    diabetes = (glu>126).astype(int)
    col = np.clip(RNG.normal(195,30,N)+0.4*(edad-45)+1.1*(imc-25)+RNG.normal(0,15,N),110,380).round(1)
    hdl = np.clip(RNG.normal(55,12,N)-0.25*(imc-25)+6*(sexo=="F")-5*fa-3*actn+RNG.normal(0,6,N),20,110).round(1)
    z = (-3.1+0.062*(edad-50)+0.019*(ta_s-120)+0.008*(col-190)-0.028*(hdl-55)+0.055*(imc-25)
         +0.85*fa+0.30*fe+0.60*diabetes+0.45*antec+0.55*(sexo=="M")
         +0.012*np.maximum(edad-65,0)*fa+RNG.normal(0,0.35,N))
    p = 1/(1+np.exp(-z))
    riesgo = (100*p).round(1); evento = RNG.binomial(1,p)
    pacientes = pd.DataFrame({"paciente_id":[f"P{i:05d}" for i in range(1,N+1)],"edad":edad,"sexo":sexo,
        "imc":imc,"ta_sistolica":ta_s,"ta_diastolica":ta_d,"glucemia":glu,"colesterol_total":col,"hdl":hdl,
        "tabaquismo":tabaquismo,"actividad_fisica":actividad,"antecedentes_familiares":antec,
        "diabetes":diabetes,"riesgo_cv_10a":riesgo,"evento_cv":evento})
    pacientes.to_csv(os.path.join(carpeta,"pacientes.csv"), index=False)

    # --- Versión "sucia" para EDA ---
    d = pacientes.copy()
    d["sexo"] = d["sexo"].map(lambda s: RNG.choice({"M":["M","m","Masculino","H","Hombre"],"F":["F","f","Femenino","Mujer"]}[s]))
    idx = RNG.choice(d.index,int(N*0.06),replace=False)
    d["glucemia"] = d["glucemia"].astype(object)   # permite mezclar texto (mmol/L con coma) y números
    d.loc[idx,"glucemia"] = [str(round(v/18.0,1)).replace(".",",") for v in d.loc[idx,"glucemia"]]
    prob_na = np.where(pacientes["edad"]<40,0.12,0.04); mask = RNG.random(N)<prob_na
    d.loc[mask,"hdl"] = np.nan
    d.loc[RNG.choice(d.index,int(N*0.05),replace=False),"colesterol_total"] = np.nan
    d.loc[RNG.choice(d.index,25,replace=False),"edad"] = RNG.integers(150,260,25)
    d.loc[RNG.choice(d.index,20,replace=False),"ta_sistolica"] = -RNG.integers(1,90,20)
    d.loc[RNG.choice(d.index,15,replace=False),"imc"] = RNG.uniform(80,200,15).round(1)
    d["colesterol_total"] = d["colesterol_total"].astype(object); d["ta_diastolica"] = d["ta_diastolica"].astype(object)
    d.loc[RNG.choice(d.index,30,replace=False),"colesterol_total"] = "desconocido"
    d.loc[RNG.choice(d.index,20,replace=False),"ta_diastolica"] = "N/D"
    d["tabaquismo"] = d["tabaquismo"].map(lambda s: RNG.choice({"nunca":["nunca","No fumador","NUNCA"],
        "ex":["ex","exfumador","Ex-fumador"],"activo":["activo","Fumador","SI"]}[s]))
    d = pd.concat([d, d.sample(400,random_state=3)], ignore_index=True)
    idx = RNG.choice(d.index,200,replace=False); d.loc[idx,"paciente_id"] = d.loc[idx,"paciente_id"].map(lambda s:" "+s+" ")
    d = d.sample(frac=1,random_state=11).reset_index(drop=True)
    d.to_csv(os.path.join(carpeta,"pacientes_sucio.csv"), index=False)

    # --- Urgencias (serie temporal) ---
    ND = 730; fechas = pd.date_range("2024-01-01", periods=ND, freq="D")
    doy = fechas.dayofyear.values; dow = fechas.dayofweek.values
    fest_set = {(1,1),(1,6),(5,1),(8,15),(10,12),(11,1),(12,6),(12,8),(12,25)}
    festivo = np.array([1 if (f.month,f.day) in fest_set else 0 for f in fechas])
    gripe = np.array([1 if f.month in (12,1,2) else 0 for f in fechas])
    temp = (15+10*np.sin(2*np.pi*(doy-110)/365)+RNG.normal(0,2.5,ND)).round(1)
    est = 1+0.14*np.sin(2*np.pi*(doy-20)/365)
    sem = np.where(dow==0,1.18,np.where(dow>=5,1.10,1.0))
    mu = 120.0*est*sem*(1+0.22*gripe)*(1+0.15*festivo)
    ingresos = RNG.poisson(np.maximum(mu,1)).astype(int)
    pd.DataFrame({"fecha":fechas.strftime("%Y-%m-%d"),"ingresos":ingresos,"festivo":festivo,
        "temporada_gripe":gripe,"temperatura":temp}).to_csv(os.path.join(carpeta,"urgencias_diarias.csv"), index=False)

    # --- Notas clínicas (texto) ---
    plant = {"cardiología":["dolor torácico opresivo de {t} de evolución, irradiado a brazo izquierdo",
        "disnea de esfuerzo progresiva y palpitaciones, antecedente de hipertensión",
        "control de anticoagulación, fibrilación auricular conocida, sin dolor actual",
        "edemas en miembros inferiores y ortopnea, sospecha de insuficiencia cardiaca"],
      "respiratorio":["tos productiva de {t}, fiebre y disnea, auscultación con crepitantes",
        "sibilancias y opresión torácica, antecedente de asma, mala respuesta a inhalador",
        "disnea súbita y dolor pleurítico, saturación disminuida",
        "tos seca persistente y febrícula, contacto con caso respiratorio"],
      "digestivo":["dolor abdominal en fosa iliaca derecha de {t}, náuseas y febrícula",
        "epigastralgia y pirosis, relación con las comidas, sin signos de alarma",
        "diarrea de {t} sin productos patológicos, buen estado general",
        "ictericia y coluria, dolor en hipocondrio derecho"],
      "neurología":["cefalea intensa de inicio súbito, la peor de su vida, con fotofobia",
        "focalidad neurológica de {t}, debilidad en hemicuerpo y disartria",
        "mareo con giro de objetos y vómitos, sin cortejo vegetativo",
        "crisis comicial presenciada, recuperación progresiva del nivel de conciencia"],
      "traumatología":["caída casual con dolor e impotencia funcional en muñeca, deformidad",
        "lumbalgia mecánica de {t} tras esfuerzo, sin déficit neurológico",
        "esguince de tobillo con edema y dificultad para la deambulación",
        "gonalgia y derrame articular tras traumatismo deportivo"]}
    prio = {"cardiología":[0.35,0.45,0.20],"respiratorio":[0.30,0.45,0.25],"digestivo":[0.20,0.50,0.30],
        "neurología":[0.45,0.35,0.20],"traumatología":[0.12,0.48,0.40]}
    tiempos = ["horas","un día","dos días","una semana","varios días"]; esp_list = list(plant.keys()); rows=[]
    for _ in range(3000):
        e = RNG.choice(esp_list); txt = RNG.choice(plant[e]).replace("{t}",RNG.choice(tiempos))
        rows.append((txt,e,RNG.choice(["alta","media","baja"],p=prio[e]),RNG.choice(centros["centro_id"].values)))
    pd.DataFrame(rows,columns=["texto","especialidad","prioridad","centro_id"]).to_csv(os.path.join(carpeta,"notas_clinicas.csv"),index=False)

    # --- Wearable (señal) ---
    rows=[]
    for s in range(1,201):
        fcb=RNG.normal(66,6); pb=RNG.normal(7500,2500); sb=RNG.normal(7.0,0.8); anom=RNG.random()<0.05
        for dia in range(1,31):
            fc=fcb+RNG.normal(0,3); pasos=max(0,pb+RNG.normal(0,1500)); sue=float(np.clip(sb+RNG.normal(0,0.6),3,11))
            if anom and dia in (14,15,16): fc+=RNG.uniform(18,30); sue-=RNG.uniform(1.5,3)
            rows.append((f"S{s:03d}",dia,round(fc,1),int(pasos),round(sue,1)))
    pd.DataFrame(rows,columns=["sujeto_id","dia","fc_reposo","pasos","horas_sueno"]).to_csv(os.path.join(carpeta,"wearable.csv"),index=False)
    return carpeta

generar_datos_clinicos(".")
print("Datos sintéticos listos en la carpeta actual. (Recuerda: son datos SINTÉTICOS, no reales.)")

## 1. El problema y la regla de oro: una partición honesta

Recordemos el escenario: la tabla `pacientes.csv` tiene 20 000 pacientes sintéticos y **dos objetivos**. Casi todo el cuaderno trabaja con `evento_cv` (0/1, **clasificación**, prevalencia ≈ 19 %); al final tocaremos `riesgo_cv_10a` (%, **regresión**).

Antes de modelar nada, la **regla sagrada** de la Unidad 3: **reservamos un `test` que no se tocará hasta el final de todo**. Ni para comparar, ni para buscar hiperparámetros. Si usáramos esos mismos datos para elegir *y* para juzgar, la cifra final **mentiría**.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pacientes = pd.read_csv("pacientes.csv")
print("Filas y columnas:", pacientes.shape)

# Variables numéricas y categóricas (según el esquema del curso)
num = ["edad", "imc", "ta_sistolica", "ta_diastolica", "glucemia",
       "colesterol_total", "hdl", "antecedentes_familiares", "diabetes"]
cat = ["sexo", "tabaquismo", "actividad_fisica"]

X = pacientes[num + cat]
y = pacientes["evento_cv"]

# Preprocesado que evita fugas: escalar numéricas (imprescindible para la SVM) y
# convertir categóricas en columnas 0/1. sparse_output=False -> matriz densa
# (la necesita HistGradientBoosting).
pre = ColumnTransformer([
    ("num", StandardScaler(), num),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat),
])

# Test RESERVADO (25 %), estratificado para conservar la prevalencia del evento.
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

print(f"Entrenamiento: {len(Xtr):>6} pacientes  | eventos: {ytr.mean():.1%}")
print(f"Test (intacto): {len(Xte):>5} pacientes  | eventos: {yte.mean():.1%}")

**Lo que hemos hecho:** hemos partido la cohorte en **entrenamiento** (15 000) y **test** (5 000), manteniendo la misma **prevalencia de eventos (≈ 19 %)** a ambos lados gracias a `stratify`. El `pre` que hemos definido **escala** las variables numéricas —clave para que la SVM no se deje engañar por que la glucemia esté en cientos y el IMC en decenas— y convierte las categóricas en 0/1.

A partir de aquí, **el test es sagrado**: no lo mirará ninguna comparación ni ninguna búsqueda. Solo lo tocaremos en el paso 4, una única vez.


## 2. Ponemos a competir a las familias (AUC en el test)

Ya tenemos modelos de sobra. Toca lo que importa: **medirlos con el mismo rasero**. Entrenamos cinco familias sobre el mismo entrenamiento y las evaluamos por **AUC-ROC** en el mismo test:

- **Regresión logística** (la campeona de la Unidad 4).
- **SVM** (`SVC` con `probability=True` para obtener probabilidades).
- **Árbol de decisión** (una sola caja blanca).
- **Random Forest** (bagging de árboles, robusto).
- **HistGradientBoosting** (boosting, el rey habitual del dato tabular).

> **Nota práctica de cómputo.** La SVM con kernel es **lenta y costosa** en cohortes grandes (su coste crece casi con el cuadrado del nº de pacientes). Para que el cuaderno corra rápido, la SVM la entrenamos sobre una **submuestra de 4 000 pacientes** del entrenamiento; las demás usan las 15 000. La evaluación se hace, para **todas**, sobre el **mismo test completo**: la comparación sigue siendo justa.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# Submuestra estratificada SOLO para entrenar la SVM (por coste de cómputo).
Xtr_svm, _, ytr_svm, _ = train_test_split(
    Xtr, ytr, train_size=4000, random_state=0, stratify=ytr)

# Definimos los cinco candidatos como pipelines (preprocesado + modelo).
modelos = {
    "Regresión logística": Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000))]),
    "SVM (RBF)":           Pipeline([("pre", pre), ("clf", SVC(probability=True, random_state=0))]),
    "Árbol de decisión":   Pipeline([("pre", pre), ("clf", DecisionTreeClassifier(max_depth=6, random_state=0))]),
    "Random Forest":       Pipeline([("pre", pre), ("clf", RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=0))]),
    "HistGradientBoosting":Pipeline([("pre", pre), ("clf", HistGradientBoostingClassifier(random_state=0))]),
}

probs_test = {}   # guardamos las probabilidades en test para reutilizarlas luego
filas = []
for nombre, modelo in modelos.items():
    if nombre.startswith("SVM"):
        modelo.fit(Xtr_svm, ytr_svm)        # SVM: submuestra de 4 000
    else:
        modelo.fit(Xtr, ytr)                # el resto: 15 000
    p = modelo.predict_proba(Xte)[:, 1]     # probabilidad de evento en el test
    probs_test[nombre] = p
    filas.append((nombre, roc_auc_score(yte, p)))

tabla_auc = (pd.DataFrame(filas, columns=["Modelo", "AUC en test"])
             .sort_values("AUC en test", ascending=False)
             .reset_index(drop=True))
tabla_auc.round(3)

**Lo que vemos (y es la lección de oro en directo):** las cinco familias quedan **muy juntas**, y la **regresión logística —el modelo más simple e interpretable— iguala o supera** a los ensembles, con un AUC en el entorno de **0,84**. ¿Por qué? Porque en estos datos el riesgo se comporta de forma aproximadamente **log-aditiva** (cada factor suma su empujón al log-odds), que es justo el terreno natural de la logística. El Random Forest y el boosting **no aportan ventaja clara** aquí: no hay interacciones fuertes que explotar. Y la SVM, además de ir por detrás, es la que **peor "da de fábrica"** y más caro cuesta entrenarla.

La moraleja: **lo complejo hay que ganárselo con números.** Aquí, no los da.


Veámoslo de un vistazo. Una barra por modelo; cuanto más alta, mejor ordena a los pacientes por su riesgo real.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
colores = ["#00AEC7", "#E4572E", "#7A5195", "#F6A600", "#4C9F70"]
ax.barh(tabla_auc["Modelo"][::-1], tabla_auc["AUC en test"][::-1],
        color=colores, edgecolor="white")
ax.set_xlim(0.5, 0.9)
ax.set_xlabel("AUC-ROC en el test (1 = perfecto, 0,5 = azar)")
ax.set_title("Comparación honesta de familias sobre evento_cv")
for i, v in enumerate(tabla_auc["AUC en test"][::-1]):
    ax.text(v + 0.003, i, f"{v:.3f}", va="center")
plt.tight_layout(); plt.show()

**Lo que vemos aquí:** todas las barras rondan un AUC parecido y la logística está **arriba del todo o empatando**. No hay un "ganador por goleada": en esta tabla clínica, el modelo simple **compite de tú a tú** con los pesos pesados. Justo lo que promete la unidad.


## 3. Validación cruzada: una estimación más honesta

La tabla anterior tiene una debilidad sutil: **una sola** partición train/test. La cifra depende de **qué pacientes cayeron por azar** en cada lado. Con suerte, en el test tocan casos fáciles y el modelo "luce"; con mala suerte, los difíciles, y parece peor.

La **validación cruzada** (*k-fold*) lo arregla: parte los datos en **k trozos** (aquí 5), entrena **k veces** reservando cada vez un trozo distinto para evaluar, y **promedia** las k cifras. El promedio es **más estable**, y su **dispersión** entre pliegues nos dice **cuánto fiarnos**.

Dos matices **clínicos** de la Unidad 3: usamos partición **estratificada** (cada pliegue conserva el ≈ 19 % de eventos) y, si un mismo paciente tuviera varias filas, **todas** deberían caer en el **mismo** pliegue (aquí cada paciente es una fila, así que no aplica). Comparamos las tres familias más rápidas sobre **toda** la cohorte.


In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

candidatos_cv = {
    "Regresión logística": modelos["Regresión logística"],
    "Random Forest":       modelos["Random Forest"],
    "HistGradientBoosting":modelos["HistGradientBoosting"],
}

for nombre, modelo in candidatos_cv.items():
    # n_jobs=1 aquí: los pliegues van en serie y cada modelo usa los núcleos.
    # (Evitamos anidar paralelismos, que en máquinas de pocos núcleos ralentiza.)
    scores = cross_val_score(modelo, X, y, cv=cv, scoring="roc_auc", n_jobs=1)
    print(f"{nombre:22s} AUC por pliegue: {np.round(scores, 3)}  "
          f"media={scores.mean():.3f}  (±{scores.std():.3f})")

**Lo que significa:** para cada modelo obtenemos **cinco** valores de AUC (uno por pliegue) y su **media ± desviación**. Dos lecturas clínicas:

- La **desviación pequeña** (unas milésimas) indica que los modelos son **estables**: no dependen de qué pacientes tocaron. Si el AUC bailara entre 0,70 y 0,88 según el pliegue, desconfiaríamos.
- Las **medias vuelven a estar muy próximas**, con la logística de nuevo a la altura de los ensembles. La validación cruzada **confirma con más solidez** lo que vimos con un solo split: aquí no hay un ganador destacado.

Esta cifra promediada es la que usaríamos para **comparar y elegir** con honestidad, mucho mejor que fiarlo todo a una única partición.


## 4. Búsqueda de hiperparámetros: afinar sin autoengañarse

Cada familia tiene "mandos" que fijamos **antes** de entrenar: la profundidad de un árbol, el nº de árboles del bosque… Son los **hiperparámetros**, y conviene no confundirlos con los **parámetros**.


{% hint style="info" %}
**Concepto · Parámetro aprendido vs hiperparámetro**

Un **parámetro** lo **aprende el modelo de los datos** durante el entrenamiento (por ejemplo, los coeficientes de la logística que se leen como *odds ratio*). Un **hiperparámetro** es una decisión de **configuración que fijamos nosotros antes** de entrenar y que el modelo no ajusta solo (la profundidad de un árbol, el nº de árboles, la tasa de aprendizaje de un boosting). Los parámetros se **aprenden**; los hiperparámetros se **eligen** —y elegirlos bien puede catapultar el rendimiento—.
{% endhint %}


Buscarlos bien es casi una **búsqueda a ciegas** (los valores por defecto rara vez son óptimos y los mandos **se afectan entre sí**), así que hay que **reentrenar con validación cruzada por cada combinación**. Usamos **`GridSearchCV`** sobre el **Random Forest** con una **rejilla PEQUEÑA** (2-3 valores por mando) para que corra rápido; `n_jobs=-1` reparte el trabajo entre núcleos. La alternativa cuando el espacio es grande sería **`RandomizedSearchCV`** (prueba combinaciones al azar).


In [ ]:
from sklearn.model_selection import GridSearchCV

# Rejilla DELIBERADAMENTE pequeña: 2 x 2 = 4 combinaciones (dos "mandos" clásicos).
rejilla = {
    "clf__n_estimators": [150, 300],
    "clf__max_depth":    [8, None],
}

# El paralelismo lo pone GridSearchCV (n_jobs=-1, reparte las combinaciones entre
# núcleos) y el bosque va en 1 hilo (n_jobs=1): así no anidamos paralelismos.
busqueda = GridSearchCV(
    Pipeline([("pre", pre), ("clf", RandomForestClassifier(n_jobs=1, random_state=0))]),
    rejilla,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=0),  # 3 pliegues: más rápido
    scoring="roc_auc",
    n_jobs=-1,
)
busqueda.fit(Xtr, ytr)   # ¡solo con el ENTRENAMIENTO! el test sigue intacto

print("Mejor combinación:", busqueda.best_params_)
print("AUC de validación de esa combinación:", round(busqueda.best_score_, 3))

**Lo que significa:** `GridSearchCV` ha probado las **4 combinaciones**, cada una evaluada con validación cruzada de 3 pliegues sobre el **entrenamiento**, y se ha quedado con la mejor. El `best_score_` es su AUC **de validación**.

{% hint style="warning" %}
**⚠️ El riesgo de "sobreajustar la validación"**

Cuidado con ese `best_score_`: es una cifra **optimista**. Si probáramos **cientos** de combinaciones y nos quedáramos con la mejor, alguna acertaría "de chiripa" en *esos* datos de validación concretos, y el número que la eligió **exageraría** lo buena que es. Por eso la única cifra en la que confiaremos de verdad es la del **test reservado**, que **no ha participado** en ninguna búsqueda, y que miramos **una sola vez** en el paso siguiente.
{% endhint %}


## 5. Elegir el mejor y evaluarlo **una sola vez** en el test

Ya tenemos el Random Forest afinado. La decisión final se toma por la **métrica clínica adecuada** (AUC, no *accuracy*) y se juzga sobre el **test intacto**, **una única vez**. `GridSearchCV` ya reentrenó el mejor modelo con todo el entrenamiento (`best_estimator_`).


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

modelo_final = busqueda.best_estimator_          # RF con los mejores hiperparámetros
prob_final = modelo_final.predict_proba(Xte)[:, 1]

auc_final = roc_auc_score(yte, prob_final)
auc_log   = roc_auc_score(yte, probs_test["Regresión logística"])
print(f"AUC en test  ·  Random Forest afinado : {auc_final:.3f}")
print(f"AUC en test  ·  Regresión logística    : {auc_log:.3f}")

**Lo que vemos:** el Random Forest afinado da un AUC honesto en el test **muy parecido** al de la logística simple. Traducción clínica: **todo el esfuerzo de búsqueda no ha comprado una ventaja real** frente al modelo humilde. En un proyecto de verdad, esto es una conclusión **valiosísima**: nos quedaríamos con la logística, que además llega con **explicabilidad de regalo** (sus *odds ratio*).


El AUC mide la capacidad de **ordenar** pacientes por riesgo, pero en la consulta hay que **decidir** con un umbral. Como el evento es raro y **no queremos que se nos escape** (coste alto del falso negativo), fijamos el umbral **sobre el entrenamiento** para lograr una **sensibilidad ≈ 85 %** y lo aplicamos al test. Así vemos sensibilidad, especificidad y valores predictivos en el punto de trabajo elegido.


In [ ]:
# Elegimos el umbral en el ENTRENAMIENTO (no en el test) para no hacer trampa.
prob_tr = modelo_final.predict_proba(Xtr)[:, 1]
objetivo_sens = 0.85
# umbral = el cuantil que deja fuera al 15 % de los eventos (sensibilidad ~85 %)
umbral = np.quantile(prob_tr[ytr == 1], 1 - objetivo_sens)

pred = (prob_final >= umbral).astype(int)
tn, fp, fn, tp = confusion_matrix(yte, pred).ravel()
sens = tp / (tp + fn)                 # sensibilidad (detecta eventos)
esp  = tn / (tn + fp)                 # especificidad (descarta sanos)
vpp  = tp / (tp + fp)                 # valor predictivo positivo
vpn  = tn / (tn + fn)                 # valor predictivo negativo

print(f"Umbral de decisión (fijado en train): {umbral:.3f}")
print(f"Sensibilidad : {sens:.1%}  (de los que tuvieron evento, cuántos detectamos)")
print(f"Especificidad: {esp:.1%}  (de los sanos, cuántos descartamos bien)")
print(f"VPP          : {vpp:.1%}  (si da positivo, prob. real de evento)")
print(f"VPN          : {vpn:.1%}  (si da negativo, prob. real de estar sano)")

**Lectura clínica:** buscábamos ~85 % de sensibilidad en el entrenamiento; en el test se materializa **algo menor (en torno al 78-80 %)** —un recordatorio de que las cifras siempre bajan un poco al pasar del train al test—, pero seguimos **detectando a la mayoría** de los que tendrán un evento. A cambio, la **especificidad** baja y el **VPP** es modesto (~40 %): con una prevalencia del 19 %, muchos avisos serán falsas alarmas. En cambio el **VPN es muy alto (~94 %)**: cuando el modelo dice "tranquilo", acierta casi siempre. Esta es exactamente la conversación que hay que tener en salud —**dónde poner el umbral es una decisión de coste clínico**, no un 0,5 por defecto—.


Terminamos el paso con la **curva ROC** de ambos, que resume el compromiso sensibilidad/especificidad para *todos* los umbrales a la vez.


In [ ]:
fpr_rf, tpr_rf, _ = roc_curve(yte, prob_final)
fpr_lg, tpr_lg, _ = roc_curve(yte, probs_test["Regresión logística"])

plt.figure(figsize=(6, 5))
plt.plot(fpr_rf, tpr_rf, color="#E4572E", label=f"Random Forest (AUC={auc_final:.2f})")
plt.plot(fpr_lg, tpr_lg, color="#00AEC7", label=f"Reg. logística (AUC={auc_log:.2f})")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Azar (AUC=0,50)")
plt.xlabel("1 - especificidad (falsos positivos)")
plt.ylabel("Sensibilidad (verdaderos positivos)")
plt.title("Curva ROC en el test: modelo elegido vs logística")
plt.legend(); plt.tight_layout(); plt.show()

**Lo que vemos aquí:** las dos curvas están **casi superpuestas** y muy por encima de la diagonal del azar. Confirmación visual de la lección de oro: para clasificar `evento_cv`, el Random Forest afinado y la logística simple son **prácticamente equivalentes**. Con criterio clínico, elegiríamos la **más simple e interpretable**.


## 6. Interpretabilidad: ¿en qué se fija el modelo?

Un modelo que nadie entiende no se puede **aprobar, auditar ni defender** ante un paciente. Empezamos por la interpretabilidad **global**: ¿qué variables pesan más en las predicciones **en conjunto**?

Usamos **`permutation_importance`**, un método **robusto** y válido para cualquier modelo: baraja al azar una columna y mide **cuánto empeora el AUC**. Si al estropear una variable el rendimiento se hunde, esa variable era **importante**. Lo hacemos sobre el **test**, con la métrica clínica (AUC).


In [ ]:
from sklearn.inspection import permutation_importance

# Sobre una submuestra del test (3 000) y 4 repeticiones: robusto y rápido.
idx_pi = Xte.sample(3000, random_state=0).index
r = permutation_importance(
    modelo_final, Xte.loc[idx_pi], yte.loc[idx_pi],
    scoring="roc_auc", n_repeats=4, random_state=0, n_jobs=-1)

importancias = (pd.DataFrame({
        "variable": (num + cat),
        "caida_de_AUC": r.importances_mean,
    })
    .sort_values("caida_de_AUC", ascending=False)
    .reset_index(drop=True))
importancias.round(4)

**Lo que vemos:** manda con diferencia la **edad**, y tras ella aparecen —como cabía esperar clínicamente— el **tabaquismo**, la **tensión sistólica** y otros factores conocidos (colesterol, HDL, IMC y **glucemia**): los grandes motores del riesgo cardiovascular. Que el modelo se apoye en las variables **correctas** es una **primera validación** de que "mira lo que debe" y no en un artefacto. Si aquí apareciera arriba una variable sin sentido (por ejemplo, un identificador), sería una señal de alarma de **fuga de datos**.


Lo dibujamos para leerlo de un vistazo.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
top = importancias.iloc[::-1]   # de menor a mayor para el gráfico horizontal
ax.barh(top["variable"], top["caida_de_AUC"], color="#00AEC7", edgecolor="white")
ax.set_xlabel("Importancia = caída de AUC al barajar la variable")
ax.set_title("Importancia de variables (permutation importance) sobre el test")
plt.tight_layout(); plt.show()

**Lo que vemos aquí:** la barra de la **edad** domina sobre todas, seguida a distancia por el **tabaquismo**, la **tensión** y el resto de factores clásicos. Coincide con el conocimiento clínico y con lo que ya nos decían los coeficientes de la logística en la Unidad 4: los modelos, por caminos distintos, **señalan a los mismos culpables**.


### SHAP (opcional): explicar la predicción de **un** paciente

La importancia global dice qué pesa **en conjunto**. **SHAP** va más allá: explica **cada predicción individual** —cuánto empuja **cada variable** el riesgo de *este* paciente hacia arriba o hacia abajo—. Es lo más "vendible" en clínica, porque convierte la caja negra en algo que **se parece a un razonamiento médico**.

La celda siguiente es **opcional** y va protegida con `try/except`: si la librería `shap` no está instalada (en Colab bastaría `!pip install shap`), el cuaderno **no se rompe** y seguimos con la alternativa robusta que ya tenemos.


In [ ]:
# --- SHAP es OPCIONAL: si no está instalado, esta celda avisa y no rompe nada ---
try:
    import shap  # puede no estar disponible en este entorno

    # SHAP sobre árboles necesita el modelo y los datos ya transformados.
    pre_fit = modelo_final.named_steps["pre"]
    clf_fit = modelo_final.named_steps["clf"]
    nombres = pre_fit.get_feature_names_out()

    # Una muestra pequeña del test, para que sea rápido.
    muestra = Xte.sample(200, random_state=0)
    muestra_T = pre_fit.transform(muestra)

    explainer = shap.TreeExplainer(clf_fit)
    sv = explainer.shap_values(muestra_T)
    sv1 = sv[1] if isinstance(sv, list) else sv   # contribuciones a la clase "evento"

    # Explicación de UN paciente concreto: variables que más le suben/bajan el riesgo.
    aportes = pd.Series(np.asarray(sv1)[0], index=nombres).sort_values(key=np.abs, ascending=False)
    print("Paciente de ejemplo — factores que más mueven su predicción (SHAP):")
    for var, val in aportes.head(6).items():
        signo = "sube" if val > 0 else "baja"
        print(f"  {var:28s} {signo} el riesgo  ({val:+.3f})")
except Exception as e:
    print("SHAP no disponible en este entorno (", type(e).__name__, ").")
    print("En Colab se instala con:  !pip install shap")
    print("Como alternativa robusta ya tenemos la importancia por permutación de arriba.")

**Lo que aporta SHAP:** si está disponible, para **un paciente concreto** produce una explicación del tipo *"su tabaquismo activo y su glucemia elevada le **suben** el riesgo; su HDL protector se lo **baja** un poco"*. Eso es justo lo que un facultativo necesita para **confiar, discutir con el paciente y documentar** la decisión —y lo que un comité o un auditor necesita para **aprobar** el sistema—. SHAP le **devuelve al ensemble** buena parte de la transparencia que había perdido frente a la logística.


## 7. Y en regresión, ¿pasa lo mismo? (spoiler: no)

Hemos visto que en la **clasificación** de `evento_cv` el modelo simple compite de tú a tú. Pero la lección de oro tiene **dos caras**. Cambiamos de tarea: ahora predecimos `riesgo_cv_10a` como **número** (regresión) y comparamos **regresión lineal** frente a **Random Forest**.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

yr = pacientes["riesgo_cv_10a"]                 # objetivo de REGRESIÓN (en %)
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(X, yr, test_size=0.25, random_state=42)

lineal = Pipeline([("pre", pre), ("clf", LinearRegression())]).fit(Xtr_r, ytr_r)
bosque = Pipeline([("pre", pre),
                   ("clf", RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=0))]).fit(Xtr_r, ytr_r)

for nombre, m in [("Regresión lineal", lineal), ("Random Forest", bosque)]:
    p = m.predict(Xte_r)
    print(f"{nombre:18s}  R² = {r2_score(yte_r, p):.3f}   MAE = {mean_absolute_error(yte_r, p):.2f} puntos")

**Lo que vemos:** aquí **cambian las tornas**. El **Random Forest gana con holgura** (R² ≈ **0,91**) a la regresión lineal (R² ≈ **0,81**), y también comete un error medio (MAE) menor. ¿Por qué ahora sí? Porque estimar el `riesgo_cv_10a` como número **tiene interacciones**: el efecto conjunto de glucemia, IMC y tabaquismo **no es una simple suma**, y el bosque **captura esas curvas e interacciones** que la recta no puede. La misma cohorte, otra tarea, **otro ganador**.


Lo confirmamos visualmente: predicho frente a real en el test. Cuanto más pegados a la diagonal, mejor.


In [ ]:
p_bosque = bosque.predict(Xte_r)
plt.figure(figsize=(5.5, 5.5))
plt.scatter(yte_r, p_bosque, s=8, alpha=0.25, color="#4C9F70")
lim = [yte_r.min(), yte_r.max()]
plt.plot(lim, lim, "--", color="gray", label="Predicción perfecta")
plt.xlabel("Riesgo real a 10 años (%)")
plt.ylabel("Riesgo predicho por el Random Forest (%)")
plt.title(f"Random Forest en regresión (R² = {r2_score(yte_r, p_bosque):.2f})")
plt.legend(); plt.tight_layout(); plt.show()

**Lo que vemos aquí:** la nube de puntos se **abraza a la diagonal**, señal de que el bosque estima el riesgo con bastante fidelidad en casi todo el rango. En regresión, la potencia extra **sí se traduce en mejores números** —justo lo contrario que en la clasificación—.


## 8. La lección de oro, ya con nuestros datos delante

{% hint style="success" %}
**💡 No hay un modelo universalmente mejor.**

El **mismo dataset** dio **ganadores distintos según la tarea**:

| Tarea | Métrica | Simple (U4) | Complejo (U5) | Ganador |
| ----- | ------- | ----------- | ------------- | ------- |
| Clasificar `evento_cv` | AUC | Logística ≈ **0,84** | Random Forest ≈ 0,83 | **Empata / gana la simple** |
| Estimar `riesgo_cv_10a` | R² | Lineal ≈ 0,81 | Random Forest ≈ **0,91** | **Gana el bosque** |

La reputación de un modelo (*"el boosting siempre gana en tabular"*) es una **hipótesis razonable, no una verdad**. La única forma honesta de elegir es **medir cada candidato con la métrica clínica adecuada, con validación cruzada, contra un baseline** —y reservar un test que no se toca hasta el final—. Ese hábito de **desconfiar de la moda y comprobar con datos** es, probablemente, lo más valioso que te llevas de todo el curso.
{% endhint %}


## 9. Qué hemos aprendido

- **Coleccionar modelos no basta; hay que medirlos bien.** Con una **partición honesta** (test reservado), la logística **igualó o superó** a SVM, árbol, Random Forest y boosting en `evento_cv`, porque el riesgo era **log-aditivo**.
- **La validación cruzada da la cifra honesta.** Un solo train/test es caprichoso; la CV promedia varias rotaciones **y te dice cuánto fiarte** (la desviación entre pliegues). En clínica: **estratificada y por paciente**.
- **Hiperparámetro ≠ parámetro.** Los parámetros se **aprenden**; los hiperparámetros se **eligen** (`GridSearchCV` / `RandomizedSearchCV`) —con una **rejilla pequeña** basta— y ojo con **sobreajustar la validación**: el test final es el único juez fiable.
- **El mejor modelo se decide por la métrica clínica** (AUC/calibración, no *accuracy*) y **una sola vez** en el test. Aquí, el RF afinado **no batió** a la logística: nos quedamos con la simple.
- **Interpretabilidad no negociable en salud.** `permutation_importance` señaló **edad, tensión, tabaquismo y glucemia**; **SHAP** (opcional) explica **paciente a paciente** y devuelve al ensemble la transparencia perdida.
- **La lección de oro tiene dos caras:** la logística ganó en clasificar `evento_cv`, pero el Random Forest **arrasó** al estimar `riesgo_cv_10a` (R² ≈ 0,91 vs 0,81). **No hay modelo universalmente mejor: se mide.**


## 10. Y esto, ¿cómo se lo pido a un asistente de IA?

Recuerda el principio del curso: **tú pones el criterio clínico, la IA escribe el código.** Un buen encargo para reproducir este cuaderno sería:

> *"Con `pacientes.csv` (datos sintéticos, target de clasificación `evento_cv`, prevalencia ≈ 19 %), en español y por celdas: (1) prepara los datos **sin fuga**, con un **test reservado** que no se toque hasta el final y escalado solo con el train (necesario para la SVM); usa una **submuestra** para la SVM si va lenta. (2) Compara con **validación cruzada estratificada** logística, SVM, Random Forest y boosting por **AUC-ROC** (media y desviación entre pliegues). (3) Afina el mejor con **GridSearchCV o RandomizedSearchCV** usando una **rejilla pequeña**, y recuérdame qué es un hiperparámetro frente a un parámetro y el riesgo de **sobreajustar la validación**. (4) Evalúa **solo al final** en el test: AUC, matriz de confusión a un umbral con **buena sensibilidad**, especificidad y VPP/VPN, y **compáralo con la logística simple**. (5) Muéstrame la **importancia de variables por permutación** y, si puedes, un **SHAP** de un paciente concreto. (6) Repite en **regresión** de `riesgo_cv_10a` (lineal vs Random Forest) y coméntame por qué cambia el ganador."*

Tu trabajo con su respuesta es el de siempre: comprobar que la **partición es honesta**, que se **compara con validación cruzada** y no con un solo split, que la **métrica destacada** encaja con el coste clínico del error (AUC/calibración, no *accuracy*), que la **búsqueda no contamina el test**, y que la conclusión **no da por ganador a un modelo por moda, sino por número**. Ese criterio es lo que no se automatiza.
